# 🛰️ NN_segmentation — Kaggle Launcher

Questo notebook è l'equivalente Kaggle del `Colab_Launcher.ipynb`.  
Funziona in modo identico ma adattato all'ambiente Kaggle.

---

## 📋 Setup iniziale (da fare UNA sola volta nel pannello Kaggle)

Prima di eseguire questo notebook, collega i dataset dal pannello **"+ Add Data"** a destra:

| Dataset Kaggle da allegare | Slug consigliato | Contiene |
|---|---|---|
| Dataset raw OpenEarthMap | `openearthmap-raw` | `open_earth_map.zip` |
| *(opzionale)* Indici subset | `oem-indices` | `oem_train_indices.json`, `oem_val_indices.json` |

### Come caricare OpenEarthMap su Kaggle
1. Vai su [kaggle.com/datasets](https://www.kaggle.com/datasets) → **New Dataset**
2. Carica il file `open_earth_map.zip`
3. Dai il nome **`openearthmap-raw`** al dataset
4. Allegalo a questo notebook dal pannello **"+ Add Data"**

### Come riusare gli indici tra sessioni
Dopo la prima esecuzione (cella ④), gli indici vengono salvati in `/kaggle/working/`.  
Per non ricalcolarli ogni volta:
1. Dal tab **"Output"** del notebook → salva `/kaggle/working/oem_train_indices.json` e `oem_val_indices.json` come nuovo Dataset Kaggle chiamato **`oem-indices`**
2. Collega quel dataset a questo notebook → dalla 2ª volta verranno caricati istantaneamente

---

## 📂 Struttura filesystem Kaggle
```
/kaggle/input/openearthmap-raw/     ← Dataset raw (read-only, allegato)
    └── open_earth_map.zip

/kaggle/input/oem-indices/          ← Indici subset (read-only, opzionale)
    ├── oem_train_indices.json
    └── oem_val_indices.json

/kaggle/working/                    ← Cartella di lavoro (read/write)
    ├── NN_segmentation/            ← Codice clonato da GitHub
    │   └── dataset/                ← Dataset estratto + JSON indici
    ├── oem_train_indices.json      ← Salvati qui per export come Dataset
    └── oem_val_indices.json
```

## ① Clone / Pull del codice da GitHub

In [ ]:
import os

REPO_URL  = 'https://github.com/robertopassante/NN_segmentation.git'
REPO_NAME = 'NN_segmentation'
WORK_DIR  = '/kaggle/working'
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)

if not os.path.exists(REPO_DIR):
    print(f'[INFO] Clonazione {REPO_NAME} in corso...')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'[INFO] Repository già presente. Aggiornamento...')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f'[OK] Working directory: {os.getcwd()}')

## ② Installazione dipendenze e pesi pre-addestrati RSP Swin-T

In [ ]:
import os

# Installa le dipendenze del progetto
!pip install -q -r requirements.txt
!pip install -q yacs safetensors

# Pesi RSP Swin-T pre-addestrati su satellitare (download solo se assenti)
WEIGHTS_PATH = '/kaggle/working/NN_segmentation/rsp-swin-t-ckpt.safetensors'
if not os.path.exists(WEIGHTS_PATH):
    print('[INFO] Download pesi RSP Swin-T...')
    !wget -q -O {WEIGHTS_PATH} \
        https://huggingface.co/BiliSakura/RSP-Swin-T/resolve/main/model.safetensors
    print('[OK] Pesi scaricati.')
else:
    print('[SKIP] Pesi RSP già presenti.')

## ③ Estrazione dataset OpenEarthMap

> Il dataset viene letto da `/kaggle/input/openearthmap-raw/` (read-only)  
> ed estratto in `/kaggle/working/NN_segmentation/dataset/` (read-write).  
> L'estrazione viene saltata se già eseguita in questa sessione.

In [ ]:
import os, shutil

# ── Configurazione percorsi ──────────────────────────────────────────────────
# Slug del dataset Kaggle allegato (come appare in /kaggle/input/)
OEM_INPUT_SLUG = 'openearthmap-raw'   # <-- modifica se hai usato un nome diverso
ZIP_NAME       = 'open_earth_map.zip'

INPUT_DIR  = f'/kaggle/input/{OEM_INPUT_SLUG}'
DATA_DIR   = '/kaggle/working/NN_segmentation/dataset'
OEM_FOLDER = os.path.join(DATA_DIR, 'OpenEarthMap')
zip_src    = os.path.join(INPUT_DIR, ZIP_NAME)
zip_dst    = os.path.join(DATA_DIR,  ZIP_NAME)

os.makedirs(DATA_DIR, exist_ok=True)

# ── Verifica che il dataset sia allegato ─────────────────────────────────────
if not os.path.exists(INPUT_DIR):
    print(f'❌ Dataset non trovato in {INPUT_DIR}')
    print(f'   → Collega il dataset "{OEM_INPUT_SLUG}" tramite il pannello "+ Add Data" di Kaggle.')
    raise FileNotFoundError(f'Dataset input mancante: {INPUT_DIR}')

# ── Estrazione (skip se già estratto) ────────────────────────────────────────
if not os.path.exists(OEM_FOLDER):
    if not os.path.exists(zip_dst):
        print(f'[INFO] Copio {ZIP_NAME} da input a working...')
        shutil.copy(zip_src, zip_dst)
        size_gb = os.path.getsize(zip_dst) / 1e9
        print(f'[OK]  Copiato ({size_gb:.1f} GB)')
    print('[INFO] Estrazione in corso (può richiedere qualche minuto)...')
    !unzip -q {zip_dst} -d {DATA_DIR}
    # Rimuoviamo il zip locale per liberare spazio (l'originale è in /input, read-only)
    os.remove(zip_dst)
    print('[OK]  Estrazione completata. ZIP locale rimosso per liberare spazio.')
else:
    print(f'[SKIP] OpenEarthMap già estratto in {OEM_FOLDER}')

%cd /kaggle/working/NN_segmentation
print('\n[SUCCESS] Dataset OpenEarthMap pronto!')

## ④ Preparazione Smart Subset (solo alla prima esecuzione)

> **Cosa fa:**  
> Analizza ogni mask → seleziona immagini con classe dominante ≥ 40%  
> (max 150/classe train, 40/classe val → ~1200+320 immagini tot)
>
> **Durata:** ~15-20 min la first run, <5 sec se hai il dataset `oem-indices` allegato.
>
> **Come rendere permanente il subset:**  
> Dopo questa cella, i JSON sono in `/kaggle/working/`.  
> Creali come Dataset Kaggle ("oem-indices") e allegali al notebook  
> → la prossima sessione questa cella durerà 2 secondi.

In [ ]:
import os, shutil

DATA_DIR     = '/kaggle/working/NN_segmentation/dataset'
WORK_DIR     = '/kaggle/working'

# Slug del dataset Kaggle degli indici (opzionale — lascia vuoto se non ce l'hai)
INDICES_SLUG = 'oem-indices'   # <-- modifica o lascia 'oem-indices'
INDICES_INPUT = f'/kaggle/input/{INDICES_SLUG}'

train_json_local = os.path.join(DATA_DIR, 'oem_train_indices.json')
val_json_local   = os.path.join(DATA_DIR, 'oem_val_indices.json')

# ── Prova a caricare da dataset Kaggle allegato (se esiste) ──────────────────
if not os.path.exists(train_json_local) and os.path.exists(INDICES_INPUT):
    for fname in ['oem_train_indices.json', 'oem_val_indices.json']:
        src = os.path.join(INDICES_INPUT, fname)
        dst = os.path.join(DATA_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, dst)
            print(f'[OK]  {fname} caricato dal dataset Kaggle "{INDICES_SLUG}".')
        else:
            print(f'[WARN] {fname} non trovato in {INDICES_INPUT}.')

# ── Verifica se ora gli indici sono disponibili ───────────────────────────────
both_exist = os.path.exists(train_json_local) and os.path.exists(val_json_local)

if both_exist:
    print('\n✅ Indici già presenti — subset pronto, skip preparazione.')
else:
    print('🔍 Avvio generazione smart subset (prima esecuzione)...')
    print('   Analisi di ogni mask con soglia 40% — circa 15-20 min')
    print('─' * 60)
    !python data/prepare_dataset.py
    print('─' * 60)

    # ── Copia i JSON anche in /kaggle/working/ per salvarli come Output ───
    for fname in ['oem_train_indices.json', 'oem_val_indices.json']:
        src = os.path.join(DATA_DIR, fname)
        dst = os.path.join(WORK_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, dst)
    
    print('\n✅ Subset pronto!')
    print('─' * 60)
    print('💡 Per non ripetere questa analisi nei prossimi notebook:')
    print('   1. Vai sul tab "Output" di questo notebook su Kaggle')
    print('   2. Trova oem_train_indices.json e oem_val_indices.json')
    print('   3. Crea un nuovo Dataset Kaggle chiamato "oem-indices" con questi file')
    print('   4. Collegalo a questo notebook da "+ Add Data"')
    print('─' * 60)

## ⑤ Training

In [ ]:
%cd /kaggle/working/NN_segmentation
!python main.py

## ⑥ Salva il modello nell'Output Kaggle

> I file in `/kaggle/working/` vengono salvati nel tab **"Output"** del notebook  
> e possono essere scaricati o usati come Dataset in altri notebook.

In [ ]:
import os, shutil

WORK_DIR = '/kaggle/working'
REPO_DIR = '/kaggle/working/NN_segmentation'

# Copia checkpoint e plot nella root di /kaggle/working/ per l'Output tab
for fname in ['best_model.pth', 'loss_curve.png']:
    src = os.path.join(REPO_DIR, fname)
    dst = os.path.join(WORK_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'[OK]  {fname} → {dst}')
    else:
        print(f'[SKIP] {fname} non trovato (training non completato?)')

# Copia la cartella delle predizioni di esempio
pred_src = os.path.join(REPO_DIR, 'output_samples')
pred_dst = os.path.join(WORK_DIR, 'output_samples')
if os.path.exists(pred_src):
    shutil.copytree(pred_src, pred_dst, dirs_exist_ok=True)
    n = len(os.listdir(pred_dst))
    print(f'[OK]  output_samples/ ({n} immagini) → {pred_dst}')

print('\n✅ Output salvati in /kaggle/working/ — visibili nel tab "Output".')